# 转录因子–靶基因调控关系抽取（改进版）

在老师示例基础上补上了 **评估 → 改 prompt → 重测 → 记录指标** 的闭环。

**运行顺序**：① 配置 → ②(可选)取文献 → ③ 抽取 → ④ 清洗 → ⑤ 建库 → ⑥ 评估。
**做迭代**：先用 `PROMPT_VERSION="v1"` 跑一遍并评估(tag=v1)，再切到 `"v2"` 重跑并评估(tag=v2)，
最后看 `runs_log.csv` 的指标对比。


## 0. 配置（路径用相对路径，跨平台；key 从环境变量读）

In [ ]:
import os
from pathlib import Path

BASE_DIR  = Path.cwd()                       # = 本 notebook 所在目录
PDF_DIR   = BASE_DIR / "实例文献"            # 把 8 篇 PDF 放这里
JSON_PATH = PDF_DIR / "extraction_results.json"
GOLD_CSV  = BASE_DIR / "gold_standard.csv"
DB_PATH   = BASE_DIR / "extraction.sqlite3"

# 在系统里设环境变量 DASHSCOPE_API_KEY，别把 key 写死在代码里
API_KEY    = os.environ.get("DASHSCOPE_API_KEY", "")
BASE_URL   = "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "qwen3.6-plus"

PROMPT_VERSION = "v2"   # 迭代开关：先跑 "v1"(老师原版) 再跑 "v2"(改进版)
print("PDF_DIR:", PDF_DIR, "| 存在:", PDF_DIR.exists())
print("当前 prompt 版本:", PROMPT_VERSION)
if not API_KEY:
    print("⚠️ 未检测到 DASHSCOPE_API_KEY，③抽取 这步会用不了（其它步骤可照常跑）")


## 1. 取文献（可选）

老师示例的检索单元是坏的（`my_tfs` 空、`alias_dict` 未定义、还残留了别的任务的
`Cell="GM12878"`），而且这 8 篇 PDF 本就是**人工下载**的。下面给一个修好的可用版本，
但默认 **跳过**——报告里写明"文献人工检索下载"即可。

In [ ]:
RUN_PUBMED_SEARCH = False    # 要自动检索就改 True
import time

TF_LIST   = ["ZBTB10"]
ALIAS_DICT = {"ZBTB10": ["ZBTB10", "RINZF"]}

def search_tf_references(tf_list):
    from Bio import Entrez
    Entrez.email = "your_email@example.com"   # 必填，否则被 NCBI 限流
    results = {}
    for tf in tf_list:
        alias_q = " OR ".join(f'"{a}"[Title/Abstract]' for a in ALIAS_DICT.get(tf, [tf]))
        query = (f"({alias_q}) AND transcription[Title/Abstract] "
                 f"AND (regulation[Title/Abstract] OR target[Title/Abstract])")
        h = Entrez.esearch(db="pubmed", term=query, retmax=50)
        rec = Entrez.read(h); h.close()
        results[tf] = rec["IdList"]
        print(f"{tf}: {len(results[tf])} 篇"); time.sleep(0.5)
    return results

if RUN_PUBMED_SEARCH:
    print(search_tf_references(TF_LIST))
else:
    print("跳过检索：使用 实例文献/ 下已有的 8 篇 PDF")


## 2. 接 API（先测连通）

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

if API_KEY:
    r = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": "只回复两个字：可用"}],
        temperature=0,
    )
    print(r.choices[0].message.content)
else:
    print("跳过：未设置 API_KEY")


## 3. 抽取（核心改进在这）

内置两套 prompt，由 `PROMPT_VERSION` 切换：
- **v1**：老师原版（允许 `Sp1, Sp3, Sp4` 这种多实体串）。
- **v2**：改进版——① 强制每条只一个 TF+一个靶基因；② 强制 HGNC 官方符号；
  ③ 收紧置信度（把"转引自他文"明确判 low），减少假阳性。

In [ ]:
import json, re
import pdfplumber

# ---------- v1：老师原版 ----------
SYS_V1 = "你是一个专业的分子生物学与生物信息学专家。请严格基于用户提供的文献内容，提取其中所有明确提及或实验验证的“转录因子调控靶基因”关系。"
USER_V1 = """
【提取要求】
1. 仅提取文献中明确陈述或提供实验证据的调控关系，禁止推测、补充外部数据库知识或合并未直接关联的实体。
2. 对每条关系提取字段（缺失填 "unknown"）：transcription_factor, target_gene, regulation_type(activation/repression/dual/unknown), evidence_type, biological_context, auxiliary_proteins, confidence(high/medium/low), location。
3. 若同一对关系多次出现，合并为一条。
4. 输出必须为纯 JSON 数组，不含任何解释或 Markdown 标记。
5. 置信度低的关系也要输出，但 confidence 注明 "low"。
【文献内容】
{pdf_text}
"""

# ---------- v2：改进版 ----------
SYS_V2 = "你是一个严谨的分子生物学与生物信息学专家。只依据用户提供的这一篇文献的正文内容，提取“转录因子调控靶基因”关系。不得引入文献之外的数据库知识，不得臆测。"
USER_V2 = """
【提取要求】
1. 仅提取本文明确陈述或提供实验证据的调控关系；不要推测、不要补充外部知识。
2. 【每条关系只能有一个转录因子和一个靶基因】。若一句涉及多个（如 Sp1/Sp3/Sp4 调控 VEGF），必须拆成多条，禁止写成 "Sp1, Sp3, Sp4"。
3. 基因名一律用 HGNC 官方 symbol（如 VEGF 写作 VEGFA），统一大小写。
4. 字段（缺失填 "unknown"）：transcription_factor(单个), target_gene(单个), regulation_type(activation/repression/dual/unknown), evidence_type(若转引自他文写 "Literature citation" 并判 low), biological_context, auxiliary_proteins(数组，未提及填 "unknown"), confidence(high=本文直接实验; medium=本文间接证据; low=仅文献引用/泛泛陈述), location。
5. 同一对关系合并为一条，保留最完整信息与最高置信度。
6. 输出必须为纯 JSON 数组，不含任何解释或 Markdown 标记。
【文献内容】
{pdf_text}
"""

PROMPTS = {"v1": (SYS_V1, USER_V1), "v2": (SYS_V2, USER_V2)}

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    return text.strip()

def parse_json_array(raw):
    raw = re.sub(r"```(?:json)?|```", "", raw).strip()
    try:
        d = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\[.*\]", raw, re.DOTALL)
        d = json.loads(m.group(0)) if m else []
    if isinstance(d, dict):
        d = d.get("relations", [d])
    return d if isinstance(d, list) else [d]

def call_model(pdf_text):
    sys_p, user_t = PROMPTS[PROMPT_VERSION]
    r = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "system", "content": sys_p},
                  {"role": "user", "content": user_t.format(pdf_text=pdf_text[:180000])}],
        temperature=0.1, max_tokens=8192,
        response_format={"type": "json_object"},
    )
    return parse_json_array(r.choices[0].message.content)

def process_one_pdf(pdf_path):
    txt = extract_text_from_pdf(pdf_path)
    if not txt:
        return {"pdf_file": pdf_path.name, "pdf_path": str(pdf_path), "status": "empty_text", "relation_count": 0, "relations": []}
    rels = call_model(txt)
    return {"pdf_file": pdf_path.name, "pdf_path": str(pdf_path), "status": "ok", "relation_count": len(rels), "relations": rels}

# 执行：遍历 PDF_DIR 下所有 PDF
if API_KEY and PDF_DIR.exists():
    pdfs = sorted(PDF_DIR.glob("*.pdf"))
    print(f"找到 {len(pdfs)} 个 PDF，用 prompt {PROMPT_VERSION} 抽取…")
    results = []
    for p in pdfs:
        try:
            res = process_one_pdf(p)
            print(f"  {p.name}: {res['status']} | {res['relation_count']} 条")
        except Exception as e:
            res = {"pdf_file": p.name, "pdf_path": str(p), "status": "failed", "relation_count": 0, "relations": [], "error": str(e)}
            print(f"  {p.name}: 失败 {e}")
        results.append(res)
    JSON_PATH.write_text(json.dumps(sorted(results, key=lambda x: x["pdf_file"]), ensure_ascii=False, indent=2), encoding="utf-8")
    print("已保存:", JSON_PATH)
else:
    print("跳过抽取（无 key 或无 PDF）。可直接用已有的 extraction_results.json 继续后面步骤。")


## 4. 清洗（拆多实体 / 统一符号 / 去重）—— 改善网站统计质量

In [ ]:
import importlib, normalize
importlib.reload(normalize)

data = json.loads(JSON_PATH.read_text(encoding="utf-8"))
clean = [normalize.clean_doc(d) for d in data if isinstance(d, dict)]
JSON_PATH.write_text(json.dumps(clean, ensure_ascii=False, indent=2), encoding="utf-8")
before = sum(len(d.get("relations", [])) for d in data)
after  = sum(d["relation_count"] for d in clean)
print(f"清洗: {before} -> {after} 条（多实体已拆开、已去重）")


## 5. 建库（写入 SQLite，网站从这里读）

In [ ]:
import json_sqlite_store as store
counts = store.rebuild_database(json_path=JSON_PATH, db_path=DB_PATH)
print("已建库:", counts)


## 6. 评估（算 P/R/F1，记录到 runs_log.csv）

⚠️ 前提：`gold_standard.csv` 要先由你**人工读 8 篇补全**，否则 precision 会被假性压低。
`TAG` 跟着 `PROMPT_VERSION` 走：跑 v1 时设 "v1"，跑 v2 时设 "v2"，方便对比。

In [ ]:
import importlib, evaluate, csv
from datetime import datetime
importlib.reload(evaluate)

TAG = PROMPT_VERSION
pred, by_pdf = evaluate.load_predicted(JSON_PATH)
gold = evaluate.load_gold(GOLD_CSV)
tp, fp, fn = pred & gold, pred - gold, gold - pred
P = len(tp)/len(pred) if pred else 0.0
R = len(tp)/len(gold) if gold else 0.0
F1 = 2*P*R/(P+R) if P+R else 0.0
print(f"[{TAG}] Precision={P:.3f}  Recall={R:.3f}  F1={F1:.3f}  (TP={len(tp)} FP={len(fp)} FN={len(fn)})")
print("FP 疑似幻觉(前15):", sorted(fp)[:15])
print("FN 漏抽(前15):", sorted(fn)[:15])

log = BASE_DIR / "runs_log.csv"
new = not log.exists()
with open(log, "a", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    if new: w.writerow(["time","tag","pred","gold","TP","FP","FN","precision","recall","f1"])
    w.writerow([datetime.now().strftime("%Y-%m-%d %H:%M"), TAG, len(pred), len(gold), len(tp), len(fp), len(fn), f"{P:.3f}", f"{R:.3f}", f"{F1:.3f}"])
print("已记录到 runs_log.csv")


## 7. 起网站

在 VS Code 终端（不是 notebook）里运行：`python app.py`，
然后打开 http://127.0.0.1:5000 ，点“重建数据库”，截图放报告。